# Purpose
Run classical ML baselines for screening/regression on AptaBench and export benchmark tables.

# Inputs
- `dataset/AptaBench_dataset.csv`
- `dataset/splits/*.json`
- Encoders from `src.encoders.*`

# Outputs
- `artifacts/ml/classification/screening/*/results.csv`
- `artifacts/ml/classification/screening/all_splits/results.csv`
- `artifacts/ml/regression/results.csv`

# Runtime (CPU/GPU, approximate)
- CPU preferred; GPU not required
- ~20-90 minutes depending on encoder set and Optuna settings

# Determinism/Seeds
- Seed controlled via global config (`SEED`) and deterministic split files.

In [ ]:
# Reviewer reproducibility config
from pathlib import Path

SEED = 42
QUICK_RUN = True
FULL_RUN = not QUICK_RUN
DEVICE = "cpu"

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PATHS = {
    "dataset_csv": REPO_ROOT / "dataset" / "AptaBench_dataset.csv",
    "splits_dir": REPO_ROOT / "dataset" / "splits",
    "ml_screening_dir": REPO_ROOT / "artifacts" / "ml" / "classification" / "screening",
    "ml_regression_dir": REPO_ROOT / "artifacts" / "ml" / "regression",
}
PATHS["ml_screening_dir"].mkdir(parents=True, exist_ok=True)
PATHS["ml_regression_dir"].mkdir(parents=True, exist_ok=True)
print({"SEED": SEED, "QUICK_RUN": QUICK_RUN, "DEVICE": DEVICE})

In [ ]:
import torch
torch.__version__

In [ ]:
# Add project root to sys.path so "src" is importable
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root.parent != root:
    root = root.parent

sys.path.insert(0, str(root))

In [ ]:
# === Imports ===
import os
import pandas as pd
from src.models.screening import screen_lgbm_optuna, load_splits
# --- Aptamer encoders ---
from src.encoders.aptamer_encoders import (
    onehot_with_type_bit,
    kmer_freq_with_type_bit,
    gena_embed,
    dnabert2_embed
)

# --- Molecule encoders ---
from src.encoders.molecule_encoders import (
    morgan_fp,
    maccs_fp,
    physchem_descriptors,
    chemberta_embed,
    unimol_embed,
    molformer_embed,
    clamp_embed
)

# Paths
DATASET_PATH = "../dataset/AptaBench_dataset.csv" 
SPLITS_DIR = "../dataset/splits"

In [ ]:
# Load dataset
df = pd.read_csv(DATASET_PATH)
print("Dataset loaded:", df.shape)
display(df.head())

# Check available splits
print("Available splits:", os.listdir(SPLITS_DIR))

## 2. Encoder configurations

In [ ]:
apt_cfgs = [
    {"name": "OneHot", "func": onehot_with_type_bit},
    {"name": "Kmer3", "func": kmer_freq_with_type_bit, "kwargs": {"k": 3}},
    {"name": "Kmer4", "func": kmer_freq_with_type_bit, "kwargs": {"k": 4}},
    {"name": "GENA", "func": gena_embed},
    {"name": "DNABERT2", "func": dnabert2_embed},
]

mol_cfgs = [
    {"name": "MorganFP", "func": morgan_fp},
    {"name": "MACCS", "func": maccs_fp},
    {"name": "PhysChem", "func": physchem_descriptors},
    {"name": "ChemBERTa", "func": chemberta_embed},
    {"name": "UniMol", "func": unimol_embed},
    {"name": "MolFormer", "func": molformer_embed},
    {"name": "CLAMP", "func": clamp_embed},
]

In [ ]:
from src.models.screening import precompute_and_save_embeddings
precompute_and_save_embeddings(df, apt_cfgs, mol_cfgs)

## 3. Classification Baselines

In [ ]:
# FAST VERSION WITH CHECKPOINTS AND INCREMENTAL SAVING
# - Checkpoints after each encoder combination
# - Automatic resume from the interruption point
# - Incremental result saving after each iteration
# - Pruning + early stopping for 2-3x speedup
# - Use all 5 folds for evaluation (no quality loss)

import gc
from pathlib import Path

# Clear memory before starting
gc.collect()

# Run one split_mode at a time
all_results = []

for split_mode in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{'='*60}")
    print(f"Processing split_mode: {split_mode}")
    print(f"{'='*60}\n")
    
    try:
        results_df = screen_lgbm_optuna(
            df,
            apt_cfgs,
            mol_cfgs,
            split_modes=(split_mode,),  # One at a time
            n_trials=10,  # Number of trials
            metric="roc_auc",
            splits_dir="../dataset/splits",
            use_cached_embeddings=True,  # Use cached embeddings
            checkpoint_dir="../artifacts/ml/classification/checkpoints",  # Checkpoint directory
            results_file=f"../artifacts/ml/classification/checkpoints/results_{split_mode}.csv",  # Incremental save file
            resume=True,  # Resume from checkpoint when available
            use_pruning=True,  # Pruning for faster search (~30-50%)
        )
        all_results.append(results_df)
        
        # Save final per-split results
        split_out = Path(f"../artifacts/ml/classification/screening/{split_mode}")
        split_out.mkdir(parents=True, exist_ok=True)
        results_df.to_csv(split_out / "results.csv", index=False)
        print(f"✓ Saved final results for {split_mode}")
        
    except Exception as e:
        print(f"✗ Error processing {split_mode}: {e}")
        import traceback
        traceback.print_exc()
        # Try to load partial results
        checkpoint_file = f"../artifacts/ml/classification/checkpoints/results_{split_mode}.csv"
        if Path(checkpoint_file).exists():
            print(f"⚠ Loading partial results from {checkpoint_file}")
            try:
                partial_df = pd.read_csv(checkpoint_file)
                all_results.append(partial_df)
            except:
                pass
    
    # Clear memory after each split_mode
    gc.collect()
    print(f"Memory cleared after {split_mode}\n")

# Combine all split results
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    combined_out = Path("../artifacts/ml/classification/screening/all_splits")
    combined_out.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(combined_out / "results.csv", index=False)
    print("✓ All results saved to ../artifacts/ml/classification/screening/all_splits/results.csv")
else:
    print("✗ No results collected")


## 4. Regression Baselines

In [ ]:
# Regression (uses embeddings loaded in cell 6)
from src.models.screening import screen_lgbm_regression_simple, load_embeddings
import gc

print(f"Regression: {df[df['pKd_value'].notna()].shape[0]}/{len(df)} samples")

# Load embeddings for the full dataset
result = load_embeddings(df, apt_cfgs, mol_cfgs, embeddings_dir="../artifacts/cache/embeddings")
if result is None:
    print("❌ Run cell 6 first!")
else:
    apt_map, mol_map = result
    print(f"✓ Loaded: {len(apt_map)} apt × {len(mol_map)} mol")
    
    gc.collect()
    results_reg = screen_lgbm_regression_simple(
        df, apt_map, mol_map,
        split_modes=["stratified", "disjoint_molecule", "disjoint_aptamer"],
        splits_dir="../dataset/splits",
    )
    
    regression_out = Path("../artifacts/ml/regression")
    regression_out.mkdir(parents=True, exist_ok=True)
    results_reg.to_csv(regression_out / "results.csv", index=False)
    print("\n✓ Saved to ../artifacts/ml/regression/results.csv")


In [ ]:
results_reg = pd.read_csv("../artifacts/ml/regression/results.csv")

In [ ]:
# Top 5 per split
for split in ["stratified", "disjoint_molecule", "disjoint_aptamer"]:
    print(f"\n{split.upper()} - Top 5 (lowest MAE):")
    top5 = results_reg[results_reg['split'] == split].nsmallest(5, 'MAE mean')
    print(top5[['aptamer_encoder', 'molecule_encoder', 'MAE mean', 'RMSE mean', 'R2 mean']].to_string(index=False))



In [ ]:
# Save under artifacts
from pathlib import Path
import shutil

print("✓ Regression results already stored in ../artifacts/ml/regression/results.csv")



In [ ]:
# Fixed-path export check (reviewer reproducibility)
from pathlib import Path

required_outputs = [
    Path("../artifacts/ml/classification/screening/all_splits/results.csv"),
    Path("../artifacts/ml/regression/results.csv"),
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected ML outputs: {missing}")
print("Verified outputs:")
for path in required_outputs:
    print(f" - {path}")